In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)

In [2]:
# Define the action
class ScalarAction(lhmc.Action):
    @staticmethod
    @jax.jit
    def _Sjax(phi, kappa, lamb):
        S = phi**2
        
        for ax in range(d):
            S -= 2 * kappa * phi * phi.nn_field(ax)

        S += lamb * (phi**2 - 1)**2    
        return S
    
    def _compute_forces(self):
        self.grads = {
            'phi': jax.jit(jax.grad(lambda phi, kappa, lamb: jnp.sum(self._Sjax(phi, kappa, lamb).F))),
        }

        def force_func(fields):
#            return {'phi': self.exact_force(fields['phi'], self.params['kappa'], self.params['lamb'])}
            return {'phi': self.grads['phi'](fields['phi'], self.params['kappa'], self.params['lamb']) }

        self.forces = force_func
    
    @staticmethod
    @jax.jit
    def exact_force(phi, kappa, lamb):
        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,d):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * kappa * J
        F += 2 * phi.F
        F += 4 * lamb * (phi.F**2 - 1) * phi.F

        return F
            

In [3]:
%%time
d = 3
Lat6 = lat.SquareLattice(dims=((6,)*d))
phi6 = lat.LatticeField(Lat6)

HMC = lhmc.HMCEvolver(
    action=ScalarAction({'phi': phi6}, params={'kappa': 0.18, 'lamb': 1.145}),
    seed=21245,
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
)

for _ in range(10000):
    HMC.evolve()

I0000 00:00:1696029622.655222       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


CPU times: user 4.28 s, sys: 156 ms, total: 4.44 s
Wall time: 4.44 s
